--- Day 20: Race Condition ---
The Historians are quite pixelated again. This time, a massive, black building looms over you - you're right outside the CPU!

While The Historians get to work, a nearby program sees that you're idle and challenges you to a race. Apparently, you've arrived just in time for the frequently-held race condition festival!

The race takes place on a particularly long and twisting code path; programs compete to see who can finish in the fewest picoseconds. The winner even gets their very own mutex!

They hand you a map of the racetrack (your puzzle input). For example:

###############
#...#...#.....#
#.#.#.#.#.###.#
#S#...#.#.#...#
#######.#.#.###
#######.#.#...#
#######.#.###.#
###..E#...#...#
###.#######.###
#...###...#...#
#.#####.#.###.#
#.#...#.#.#...#
#.#.#.#.#.#.###
#...#...#...###
###############
The map consists of track (.) - including the start (S) and end (E) positions (both of which also count as track) - and walls (#).

When a program runs through the racetrack, it starts at the start position. Then, it is allowed to move up, down, left, or right; each such move takes 1 picosecond. The goal is to reach the end position as quickly as possible. In this example racetrack, the fastest time is 84 picoseconds.

Because there is only a single path from the start to the end and the programs all go the same speed, the races used to be pretty boring. To make things more interesting, they introduced a new rule to the races: programs are allowed to cheat.

The rules for cheating are very strict. Exactly once during a race, a program may disable collision for up to 2 picoseconds. This allows the program to pass through walls as if they were regular track. At the end of the cheat, the program must be back on normal track again; otherwise, it will receive a segmentation fault and get disqualified.

So, a program could complete the course in 72 picoseconds (saving 12 picoseconds) by cheating for the two moves marked 1 and 2:

###############
#...#...12....#
#.#.#.#.#.###.#
#S#...#.#.#...#
#######.#.#.###
#######.#.#...#
#######.#.###.#
###..E#...#...#
###.#######.###
#...###...#...#
#.#####.#.###.#
#.#...#.#.#...#
#.#.#.#.#.#.###
#...#...#...###
###############
Or, a program could complete the course in 64 picoseconds (saving 20 picoseconds) by cheating for the two moves marked 1 and 2:

###############
#...#...#.....#
#.#.#.#.#.###.#
#S#...#.#.#...#
#######.#.#.###
#######.#.#...#
#######.#.###.#
###..E#...12..#
###.#######.###
#...###...#...#
#.#####.#.###.#
#.#...#.#.#...#
#.#.#.#.#.#.###
#...#...#...###
###############
This cheat saves 38 picoseconds:

###############
#...#...#.....#
#.#.#.#.#.###.#
#S#...#.#.#...#
#######.#.#.###
#######.#.#...#
#######.#.###.#
###..E#...#...#
###.####1##.###
#...###.2.#...#
#.#####.#.###.#
#.#...#.#.#...#
#.#.#.#.#.#.###
#...#...#...###
###############
This cheat saves 64 picoseconds and takes the program directly to the end:

###############
#...#...#.....#
#.#.#.#.#.###.#
#S#...#.#.#...#
#######.#.#.###
#######.#.#...#
#######.#.###.#
###..21...#...#
###.#######.###
#...###...#...#
#.#####.#.###.#
#.#...#.#.#...#
#.#.#.#.#.#.###
#...#...#...###
###############
Each cheat has a distinct start position (the position where the cheat is activated, just before the first move that is allowed to go through walls) and end position; cheats are uniquely identified by their start position and end position.

In this example, the total number of cheats (grouped by the amount of time they save) are as follows:

There are 14 cheats that save 2 picoseconds.
There are 14 cheats that save 4 picoseconds.
There are 2 cheats that save 6 picoseconds.
There are 4 cheats that save 8 picoseconds.
There are 2 cheats that save 10 picoseconds.
There are 3 cheats that save 12 picoseconds.
There is one cheat that saves 20 picoseconds.
There is one cheat that saves 36 picoseconds.
There is one cheat that saves 38 picoseconds.
There is one cheat that saves 40 picoseconds.
There is one cheat that saves 64 picoseconds.
You aren't sure what the conditions of the racetrack will be like, so to give yourself as many options as possible, you'll need a list of the best cheats. How many cheats would save you at least 100 picoseconds?

In [58]:
sample_maze_str = """###############
#...#...#.....#
#.#.#.#.#.###.#
#S#...#.#.#...#
#######.#.#.###
#######.#.#...#
#######.#.###.#
###..E#...#...#
###.#######.###
#...###...#...#
#.#####.#.###.#
#.#...#.#.#...#
#.#.#.#.#.#.###
#...#...#...###
###############"""
sample_maze = [line for line in sample_maze_str.splitlines()]

def find_symbols(symbol, maze):
    findings = set()
    for i, line in enumerate(maze):
        for j, char in enumerate(line):
            if char == symbol:
                findings.add((i, j))
    if len(findings) == 1:
        findings = findings.pop()
    return findings

start = find_symbols('S', sample_maze)
end = find_symbols('E', sample_maze)
track = find_symbols('.', sample_maze)
track.update((start, end))

def neighbors(loc, track):
    i, j = loc
    candidates = set([(i, j+1), (i, j-1), (i+1, j), (i-1,j)])
    return candidates & track

def get_next(current, track, visited):
    for n in neighbors(current, track):
        if n not in visited:
            return n

track_time = {start : 0}
current = start
for i in range(1, len(track) + 1):
    if next_track := get_next(current, track, track_time):
        track_time[next_track] = i
        current = next_track
    else:
        break

def distance(l1, l2): #L1 distance
     delta = [abs(i - j) for i, j in zip(l1, l2)]
     return sum(delta)

num_cheats = {}
for loc_1, t_1 in track_time.items():
    for loc_2, t_2 in track_time.items():
        if t_2 > t_1 + 1:
            if distance(loc_1, loc_2) == 2 and (t_2 - t_1) > 2:
                saved = t_2 - t_1 - distance(loc_1, loc_2)
                num_cheats[saved] = num_cheats.setdefault(saved, 0) + 1
dict(sorted(num_cheats.items()))

{2: 14, 4: 14, 6: 2, 8: 4, 10: 2, 12: 3, 20: 1, 36: 1, 38: 1, 40: 1, 64: 1}

In [60]:
# let's also do part 2 for the sample input here:

num_cheats = {}
for loc_1, t_1 in track_time.items():
    for loc_2, t_2 in track_time.items():
        if t_2 > t_1:
            dist = distance(loc_1, loc_2)
            if 2 <= dist <= 20:
                saved_time = t_2 - t_1 - dist
                if saved_time >= 50:
                    num_cheats[saved_time] = num_cheats.setdefault(saved_time, 0) + 1
dict(sorted(num_cheats.items()))

{50: 32,
 52: 31,
 54: 29,
 56: 39,
 58: 25,
 60: 23,
 62: 20,
 64: 19,
 66: 12,
 68: 14,
 70: 12,
 72: 22,
 74: 4,
 76: 3}

In [48]:
# It looks like the cheat can only jump a single wall, distance between start and end tile (which are both on track) needs to be exactly 2
# Ok let's do it with the real puzzle input:
with open('input20.txt') as file:
    data = [line.rstrip() for line in file]
start = find_symbols('S', data)
end = find_symbols('E', data)
track = find_symbols('.', data)
track.update((start, end))
start, end, len(track)

((79, 59), (87, 39), 9357)

In [49]:
track_time = {start : 0}
current = start
for i in range(1, len(track) + 1):
    if next_track := get_next(current, track, track_time):
        track_time[next_track] = i
        current = next_track
    else:
        break
len(track_time)

9357

In [51]:
num_cheats = 0
for loc_1, t_1 in track_time.items():
    for loc_2, t_2 in track_time.items():
        if t_2 > t_1 + 100:
            dist = distance(loc_1, loc_2)
            if dist == 2:
                saved_time = t_2 - t_1 - dist
                if saved_time >= 100:
                    num_cheats += 1
num_cheats

1323

Your puzzle answer was 1323.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
The programs seem perplexed by your list of cheats. Apparently, the two-picosecond cheating rule was deprecated several milliseconds ago! The latest version of the cheating rule permits a single cheat that instead lasts at most 20 picoseconds.

Now, in addition to all the cheats that were possible in just two picoseconds, many more cheats are possible. This six-picosecond cheat saves 76 picoseconds:

###############
#...#...#.....#
#.#.#.#.#.###.#
#S#...#.#.#...#
#1#####.#.#.###
#2#####.#.#...#
#3#####.#.###.#
#456.E#...#...#
###.#######.###
#...###...#...#
#.#####.#.###.#
#.#...#.#.#...#
#.#.#.#.#.#.###
#...#...#...###
###############
Because this cheat has the same start and end positions as the one above, it's the same cheat, even though the path taken during the cheat is different:

###############
#...#...#.....#
#.#.#.#.#.###.#
#S12..#.#.#...#
###3###.#.#.###
###4###.#.#...#
###5###.#.###.#
###6.E#...#...#
###.#######.###
#...###...#...#
#.#####.#.###.#
#.#...#.#.#...#
#.#.#.#.#.#.###
#...#...#...###
###############
Cheats don't need to use all 20 picoseconds; cheats can last any amount of time up to and including 20 picoseconds (but can still only end when the program is on normal track). Any cheat time not used is lost; it can't be saved for another cheat later.

You'll still need a list of the best cheats, but now there are even more to choose between. Here are the quantities of cheats in this example that save 50 picoseconds or more:

There are 32 cheats that save 50 picoseconds.
There are 31 cheats that save 52 picoseconds.
There are 29 cheats that save 54 picoseconds.
There are 39 cheats that save 56 picoseconds.
There are 25 cheats that save 58 picoseconds.
There are 23 cheats that save 60 picoseconds.
There are 20 cheats that save 62 picoseconds.
There are 19 cheats that save 64 picoseconds.
There are 12 cheats that save 66 picoseconds.
There are 14 cheats that save 68 picoseconds.
There are 12 cheats that save 70 picoseconds.
There are 22 cheats that save 72 picoseconds.
There are 4 cheats that save 74 picoseconds.
There are 3 cheats that save 76 picoseconds.
Find the best cheats using the updated cheating rules. How many cheats would save you at least 100 picoseconds?



In [55]:
# Unchanged from Part 1:

with open('input20.txt') as file:
    data = [line.rstrip() for line in file]
start = find_symbols('S', data)
end = find_symbols('E', data)
track = find_symbols('.', data)
track.update((start, end))

track_time = {start : 0}
current = start
for i in range(1, len(track) + 1):
    if next_track := get_next(current, track, track_time):
        track_time[next_track] = i
        current = next_track
    else:
        break

In [56]:
num_cheats = 0
for loc_1, t_1 in track_time.items():
    for loc_2, t_2 in track_time.items():
        if t_2 > t_1 + 100:
            dist = distance(loc_1, loc_2)
            if 2 <= dist <= 20:
                saved_time = t_2 - t_1 - dist
                if saved_time >= 100:
                    num_cheats += 1
num_cheats

983905

That's the right answer! You are one gold star closer to finding the Chief Historian.